# 2. What did the agent learn?

1. **How strong is it?** Matches against simple players, with and without search.
2. **Which first move does it prefer?** Connect-4 is solved: with perfect play, the first player wins only by starting in the centre column.
3. **What is inside the network?** Probes test whether three simple concepts can be read from each layer:
   - `can_win_now`: the player to move can win with their next move
   - `must_block`: the opponent threatens to win on their next move
   - `controls_centre`: the player to move has more stones in the centre column

   Each probe is compared with the same probe on an **untrained** network, to separate what training added from what random features already give.

Run notebook 1 first.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/meg-x08/connect4-alphazero-xai.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Save checkpoints and results to Google Drive so they survive a disconnect
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/connect4-zero"
    REPO_DIR = "/content/repo"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    # Running locally from the notebooks/ folder
    REPO_DIR = os.path.abspath("..")
    WORK_DIR = REPO_DIR

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
CHECKPOINT_DIR = os.path.join(WORK_DIR, "checkpoints")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("checkpoints:", CHECKPOINT_DIR)
print("results:    ", RESULTS_DIR)

In [ ]:
import glob, json
import matplotlib.pyplot as plt
import numpy as np
import torch

from connect4.game import Connect4
from connect4.mcts import mcts
from connect4.network import load_network, save_checkpoint
from connect4.players import AlphaZeroPlayer, GreedyPlayer, NetworkPlayer, RandomPlayer, play_match
from connect4.probing import collect_positions, concept_labels, layer_features, probe_all, probe_score

checkpoints = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "iter_*.pt")))
print(f"{len(checkpoints)} checkpoints, using {os.path.basename(checkpoints[-1])}")
net = load_network(checkpoints[-1])
untrained = load_network(checkpoints[0])
results = {}

## 1. How strong is it?

"alphazero" is the full agent (network + 50 search simulations per move). "network only" plays the network's top move with no search.

Each match is 100 games and the players take turns starting. Even with 100 games, a win rate can change by a few percent between runs.

In [ ]:
matches = []
for player in (AlphaZeroPlayer(net, num_simulations=50), NetworkPlayer(net)):
    for opponent in (RandomPlayer(), GreedyPlayer()):
        wins, draws, losses = play_match(player, opponent, num_games=100)
        matches.append({"player": player.name, "opponent": opponent.name,
                        "wins": wins, "draws": draws, "losses": losses})
        print(f"{player.name:22s} vs {opponent.name:7s}: {wins} wins, {draws} draws, {losses} losses")
results["matches"] = matches

## 2. Which first move does it prefer?

In [ ]:
policy, value = net.predict(Connect4())
visits = mcts(Connect4(), net, num_simulations=400)
columns = np.arange(1, 8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
ax1.bar(columns, policy)
ax1.set_title("Network only: move probabilities")
ax2.bar(columns, visits / visits.sum(), color="tab:orange")
ax2.set_title("With search: share of 400 simulations")
for ax in (ax1, ax2):
    ax.set_xticks(columns)
    ax.set_xlabel("column")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "opening.png"), dpi=150)
plt.show()
results["opening_policy"] = policy.round(3).tolist()
results["opening_search"] = (visits / visits.sum()).round(3).tolist()

## 3. What is inside the network?

Positions come from games between random and greedy players. For every layer, a logistic-regression probe is trained on positions from 80% of the games and tested on the other 20%.
The score is **balanced accuracy**: 0.5 means guessing, 1.0 means perfect.

In [ ]:
positions, game_ids = collect_positions(num_games=400)
labels = concept_labels(positions)
print(f"{len(positions)} positions")
for name, y in labels.items():
    print(f"  {name}: true in {y.mean():.0%} of positions")

In [ ]:
trained_scores = probe_all(net, positions, labels, game_ids)
untrained_scores = probe_all(untrained, positions, labels, game_ids)

layers = list(next(iter(trained_scores.values())))
fig, axes = plt.subplots(1, len(labels), figsize=(14, 4), sharey=True)
for ax, concept in zip(axes, labels):
    ax.plot(layers, [untrained_scores[concept][l] for l in layers], "o-", color="gray", label="untrained network")
    ax.plot(layers, [trained_scores[concept][l] for l in layers], "o-", color="tab:blue", label="trained network")
    ax.axhline(0.5, ls="--", color="lightgray")
    ax.set_title(concept)
    ax.set_ylim(0.4, 1.02)
axes[0].set_ylabel("probe balanced accuracy")
axes[0].legend()
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "probes.png"), dpi=150)
plt.show()
results["probes_trained"] = trained_scores
results["probes_untrained"] = untrained_scores

### Does the concept appear during training?

The same probes on the last layer (`conv4`) for networks saved at different iterations.

In [ ]:
picked = [c for c in checkpoints if int(os.path.basename(c)[5:8]) in (0, 5, 10, 20, 30, 40)]
history = {concept: [] for concept in labels}
iterations = []
for path in picked:
    features = layer_features(load_network(path), positions)["conv4"]
    iterations.append(int(os.path.basename(path)[5:8]))
    for concept, y in labels.items():
        history[concept].append(probe_score(features, y, game_ids))

fig, ax = plt.subplots(figsize=(6, 4))
for concept, scores in history.items():
    ax.plot(iterations, scores, "o-", label=concept)
ax.axhline(0.5, ls="--", color="lightgray")
ax.set_xlabel("training iteration")
ax.set_ylabel("probe balanced accuracy (conv4)")
ax.set_ylim(0.4, 1.02)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "probes_over_training.png"), dpi=150)
plt.show()
results["probes_over_training"] = {"iterations": iterations, **history}

## Save the results

In [ ]:
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
save_checkpoint(os.path.join(RESULTS_DIR, "model.pt"), net)
print("saved to", RESULTS_DIR)

## Play against the agent

You are X and move first (set `YOU = -1` to let the agent start). Type a column from 1 to 7.

In [ ]:
YOU = 1
game = Connect4()
agent = AlphaZeroPlayer(net, num_simulations=200)
while not game.is_over():
    print(game, "\n")
    if game.current_player == YOU:
        move = int(input("your move (1-7): ")) - 1
    else:
        move = agent.choose_move(game)
        print(f"agent plays {move + 1}")
    game.play(move)
print(game)
print({1: "X wins", -1: "O wins", 0: "draw"}[game.winner])